In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import time


from utils_ML import score_dataset
from utils_preproces import ols_fit, y_distribution_plot, corr_heatmap_plot, make_mi_scores, preprocess_data

from sklearn.ensemble import RandomForestRegressor, VotingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor, plot_importance
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold


from sklearn.metrics import mean_absolute_error

In [33]:
# Set Matplotlib defaults
plt.style.use("seaborn-v0_8-white")
plt.rc("figure", autolayout=True)
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=14,
    titlepad=10,
)

In [34]:
# Fix random state
random_state = 42

In [35]:
harps = pd.read_csv('harps_sample.csv')
apogee = pd.read_csv('apogee_sample.csv')
galah = pd.read_csv('galah_sample.csv')
gr8 = pd.read_csv('gr8_not_in_harps_sample.csv')

print (f'harps contains {len(harps)} stars')
print (f'apogee contains {len(apogee)} stars')    
print (f'galah contains {len(galah)} stars')
print (f'gr8 contains {len(gr8)} stars')

harps contains 798 stars
apogee contains 95586 stars
galah contains 116050 stars
gr8 contains 1430 stars


In [36]:
apogee = apogee[apogee.parallax >2]
print (len(apogee))
galah = galah[galah.parallax > 2]
print (len(galah))


20482
45103


In [37]:
def fit_trasform(X):
    X['VW_lsr'] = np.sqrt(X['V_lsr']**2 + X['W_lsr']**2)
    X['UW_lsr'] = np.sqrt(X['U_lsr']**2 + X['W_lsr']**2)
    X['UV_lsr'] = np.sqrt(X['U_lsr']**2 + X['V_lsr']**2)
    X['UVW_lsr'] = np.sqrt(X['U_lsr']**2 + X['V_lsr']**2 + X['W_lsr']**2)
    epsilon = 1e-6
    #log features
    #X['Log_RV'] = np.log(X['RV'] + np.abs(X['RV'].min()))
    #X['Log_U_lsr'] = np.log(X['U_lsr'] + np.abs(X['U_lsr'].min()) + epsilon)
    #X['Log_V_lsr'] = np.log(X['V_lsr'] + np.abs(X['V_lsr'].min()) + epsilon)
    X['abs_W_lsr'] =np.abs(X['W_lsr'])
    X['abs_U_lsr'] =np.abs(X['U_lsr'])
    #X['Log_UW_lsr'] = np.log(X['UW_lsr'])
    #X['Log_UVW_lsr'] = np.log(X['UVW_lsr'])
    X.drop(columns = ['U_lsr', 'W_lsr'], inplace = True)
    #X['R_mean'] = (X['rapo'] + X['rperi'])/2
    #X.drop(columns = ['rperi', 'rapo'], inplace = True)

    return X

apogee = fit_trasform(apogee)
harps = fit_trasform(harps)
galah = fit_trasform(galah)
gr8 = fit_trasform(gr8)

# Here we focus on getting Fe from no abundance

In [38]:
columns_for_Fe = ['P_gal', 'VW_lsr', 'V_lsr', 'zmax', 'rperi', 'abs_W_lsr', 'rapo', 'abs_U_lsr',
                'fe_h', ]

apogee = apogee[columns_for_Fe]
harps = harps[columns_for_Fe]
galah = galah[columns_for_Fe]
gr8 = gr8[columns_for_Fe]

### Identifying and exluding null values

In [39]:
apogee.isnull().sum() # 196 null value for P_gal
apogee.dropna(inplace = True)

In [40]:
galah.isnull().sum() # 1 null value for P_gal
galah.dropna(inplace = True)

In [41]:
#gr8.isnull().sum() # No null values

In [42]:
#harps.isnull().sum() # No null values

### Spliting data into Trianing and Testing sets

In [43]:
# Split APOGEE
apogee_train_full, apogee_test = train_test_split(apogee, test_size=0.1, random_state=random_state)
# Split GALAH
galah_train_full, galah_test = train_test_split(galah, test_size=0.3, random_state=random_state)
# Combine APOGEE and GALAH train sets
combined_train = pd.concat([apogee_train_full], ignore_index=True)


In [44]:

import h2o
from h2o.automl import H2OAutoML

h2o.init()

train_h2o = h2o.H2OFrame(combined_train)
test_h2o = h2o.H2OFrame(harps)

y_col = 'fe_h'  
x_cols = [col for col in train_h2o.columns if col != y_col]

aml = H2OAutoML(max_runtime_secs=1200, max_models=20, seed=random_state)
aml.train(x=x_cols, y=y_col, training_frame=train_h2o)

pred = aml.leader.predict(test_h2o).as_data_frame().values.flatten()
print("H2O AutoML MAE:", mean_absolute_error(harps['fe_h'], pred))



Checking whether there is an H2O instance running at http://localhost:54321. connected.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,2 hours 28 mins
H2O_cluster_timezone:,Europe/Lisbon
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.7
H2O_cluster_version_age:,7 months and 2 days
H2O_cluster_name:,H2O_from_python_vadibekyan_36kyyy
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,6.734 Gb
H2O_cluster_total_cores:,20
H2O_cluster_allowed_cores:,20
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%
H2O AutoML MAE: 0.17261496331893955


/home/vadibekyan/anaconda3/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [45]:
# Show the leaderboard
lb = aml.leaderboard
leaderboard_df = lb.as_data_frame()
leaderboard_df


/home/vadibekyan/anaconda3/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


,model_id,rmse,mse,mae,rmsle,mean_residual_deviance
0,StackedEnsemble_AllModels_1_AutoML_2_20251029_163611,0.174381,0.030409,0.136077,0.214269,0.030409
1,StackedEnsemble_BestOfFamily_1_AutoML_2_20251029_163611,0.174398,0.030415,0.136143,0.214218,0.030415
2,GBM_5_AutoML_2_20251029_163611,0.174861,0.030576,0.136426,0.214804,0.030576
3,GBM_2_AutoML_2_20251029_163611,0.174980,0.030618,0.136517,0.214975,0.030618
4,GBM_1_AutoML_2_20251029_163611,0.175050,0.030642,0.136539,0.214909,0.030642
5,GBM_3_AutoML_2_20251029_163611,0.175171,0.030685,0.136731,0.215151,0.030685
6,GBM_grid_1_AutoML_2_20251029_163611_model_1,0.175361,0.030752,0.136810,0.215263,0.030752
7,DeepLearning_grid_3_AutoML_2_20251029_163611_model_1,0.175399,0.030765,0.136691,NaN,0.030765
8,DeepLearning_grid_2_AutoML_2_20251029_163611_model_1,0.175426,0.030774,0.136951,NaN,0.030774
9,DeepLearning_grid_1_AutoML_2_20251029_163611_model_1,0.175713,0.030875,0.137074,NaN,0.030875


In [46]:
#best_model = h2o.get_model(leaderboard_df.loc[0, 'model_id'])
#best_model.params
#aml.leader #gives the best model
# Plot variable importance
#best_model.varimp_plot()

In [47]:
from autogluon.tabular import TabularPredictor

# Combine features and target into one DataFrame
train_df = combined_train
test_df = harps

# Create predictor
predictor = TabularPredictor(label='fe_h', eval_metric='mae').fit(train_df, time_limit=4*3600)
# Predict on test set
y_pred = predictor.predict(test_df)

# Evaluate
from sklearn.metrics import mean_absolute_error
print("AutoGluon MAE:", mean_absolute_error(harps['fe_h'], y_pred))


No path specified. Models will be saved in: "AutogluonModels/ag-20251029_164052"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #33~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Fri Sep 19 17:02:30 UTC 2
CPU Count:          20
Memory Avail:       14.29 GB / 31.05 GB (46.0%)
Disk Space Avail:   516.60 GB / 936.79 GB (55.1%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. Recommended for most users. Use

	If 'regression' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    14626.98 MB
	Train Data (Original)  Memory Usage: 1.13 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', 

AutoGluon MAE: 0.1727184217901047


In [48]:
predictor.leaderboard(silent=False) # shows performance of the models

                  model  score_val          eval_metric  pred_time_val    fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0   WeightedEnsemble_L2  -0.135652  mean_absolute_error       0.299538  147.004800                0.000997           0.090300            2       True         12
1              CatBoost  -0.136105  mean_absolute_error       0.004388    3.590750                0.004388           3.590750            1       True          6
2              LightGBM  -0.136208  mean_absolute_error       0.011155    5.077062                0.011155           5.077062            1       True          4
3            LightGBMXT  -0.136210  mean_absolute_error       0.015598    6.412941                0.015598           6.412941            1       True          3
4        NeuralNetTorch  -0.136232  mean_absolute_error       0.027570  113.183444                0.027570         113.183444            1       True         10
5       NeuralNetFastAI  -0.136737

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.135652,mean_absolute_error,0.299538,147.004800,0.000997,0.090300,2,True,12
1,CatBoost,-0.136105,mean_absolute_error,0.004388,3.590750,0.004388,3.590750,1,True,6
2,LightGBM,-0.136208,mean_absolute_error,0.011155,5.077062,0.011155,5.077062,1,True,4
3,LightGBMXT,-0.136210,mean_absolute_error,0.015598,6.412941,0.015598,6.412941,1,True,3
4,NeuralNetTorch,-0.136232,mean_absolute_error,0.027570,113.183444,0.027570,113.183444,1,True,10
5,NeuralNetFastAI,-0.136737,mean_absolute_error,0.038868,23.193302,0.038868,23.193302,1,True,8
6,XGBoost,-0.136880,mean_absolute_error,0.006413,1.900222,0.006413,1.900222,1,True,9
7,LightGBMLarge,-0.136881,mean_absolute_error,0.012418,17.335771,0.012418,17.335771,1,True,11
8,ExtraTreesMSE,-0.137681,mean_absolute_error,0.216560,1.869942,0.216560,1.869942,1,True,7
9,RandomForestMSE,-0.139396,mean_absolute_error,0.240464,10.577157,0.240464,10.577157,1,True,5


In [49]:
leaderboard_df = predictor.leaderboard(silent=True)
best_model = leaderboard_df.iloc[0]['model']
print("Best model:", best_model)
y_pred = predictor.predict(test_df, model=best_model)
print("MAE:", mean_absolute_error(test_df['fe_h'], y_pred))
performance = predictor.evaluate_predictions(
    y_true=test_df['fe_h'],
    y_pred=predictor.predict(test_df, model='XGBoost'),
    auxiliary_metrics=True
)
print(performance)

Best model: WeightedEnsemble_L2
MAE: 0.1727184217901047
{'mean_absolute_error': -0.17248003503701734, 'root_mean_squared_error': np.float64(-0.21889125189955866), 'mean_squared_error': -0.04791338015815604, 'r2': 0.28784967289643315, 'pearsonr': 0.5824842437514759, 'median_absolute_error': -0.1420016708225012}
